In [1]:
import sys
!{sys.executable} -m pip install datasets

  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/559.1 kB ? eta -:--:--
   ---------------------------------------- 559.1/559.1 kB 9.4 MB/s  0:00:00
   ---------------------------------------- 0.0/28.6 MB ? eta -:--:--
   --- ------------------------------------ 2.4/28.6 MB 12.0 MB/s eta 0:00:03
   ------ --------------------------------- 5.0/28.6 MB 11.8 MB/s eta 0:00:03
   ---------- ----------------------------- 7.3/28.6 MB 11.8 MB/s eta 0:00:02
   ------------- -------------------------- 9.7/28.6 MB 11.8 MB/s eta 0:00:02
   ----------------- ---------------------- 12.3/28.6 MB 11.8 MB/s eta 0:00:02
   ------------------- -------------------- 14.2/28.6 MB 11.3 MB/s eta 0:00:02
   --------------------- ------------------ 15.7/28.6 MB 10.7 MB/s eta 0:00:02
   ------------------------ --------------- 17.8/28.6 MB 10.5 MB/s eta 0:00:02
   --------------------------- ------------ 19.4/28.6 MB 10.4 MB/s eta 0:00:01
   ----


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from datasets import load_dataset

print("Downloading Legal RAG Bench from Hugging Face...")
# We load the dataset. HF datasets often have different "splits" (train, dev, test)
dataset = load_dataset("isaacus/legal-rag-bench")

print("\nDownload complete! Here is the structure of the dataset:")
print(dataset)

C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\agrim\PycharmProjects\pilotproject-deepfix\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\agrim\.cache\huggingface\hub\datasets--isaacus--legal-rag-bench. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 4876/4876 [00:00<00:00, 92180.84 examples/s]


Download complete! Here is the structure of the dataset:
DatasetDict({
    test: Dataset({
        features: ['id', 'title', 'text', 'footnotes'],
        num_rows: 4876
    })
})


In [4]:
from datasets import load_dataset
from llama_index.core import Document

# 1. Download both parts of the dataset specifically
print("Downloading the Corpus (the legal documents)...")
corpus_dataset = load_dataset("isaacus/legal-rag-bench", "corpus", split="test")

print("Downloading the QA pairs (the questions and correct answers)...")
qa_dataset = load_dataset("isaacus/legal-rag-bench", "qa", split="test")

print(f"\nSuccess! Loaded {len(corpus_dataset)} documents and {len(qa_dataset)} questions.")

# 2. Convert HuggingFace data into LlamaIndex Documents
print("\nConverting corpus to LlamaIndex format...")
llama_documents = []
for row in corpus_dataset:
    # We take the text, and save the ID and Title as "metadata" so the AI knows where it came from
    doc = Document(text=row["text"], metadata={"id": row["id"], "title": row["title"]})
    llama_documents.append(doc)

print("Conversion complete! The RAG system is ready to index these documents.")

# 3. Let's peek at the first question Tetyana will use to test our RAG
print("\n--- Let's look at the first test question ---")
print(qa_dataset[0])

Generating test split: 100%|██████████| 100/100 [00:00<00:00, 2033.27 examples/s]



Success! Loaded 4876 documents and 100 questions.

Converting corpus to LlamaIndex format...
Conversion complete! The RAG system is ready to index these documents.

--- Let's look at the first test question ---
{'id': 1, 'question': 'Bob and Ted are close friends. Ted is on trial for drug offences, and Bob has been selected as a juror in Ted’s case. Is the judge required to excuse Bob from serving on the jury?', 'answer': 'No. While the bench book instructs judges to inform members of the jury panel that they can excuse themselves if they know the accused, this is not mandatory. Instead, the court may excuse a potential juror if they are satisfied that the person will not be able to consider the case impartially.', 'relevant_passage_id': '1.2-c2-s2'}


In [5]:
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# 1. Set up the embedding model and disable the LLM for now
print("Setting up embedding model...")
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = None

# 2. Build the Vector Index using the 4,876 legal documents!
print("Building the Vector Index (This will take a minute or two)...")
index = VectorStoreIndex.from_documents(llama_documents)
print("Index built successfully!")

# 3. Create the retriever
retriever = index.as_retriever(similarity_top_k=3) # Let's get the top 3 best matches

# 4. Test it with the very first question from the QA dataset
test_question = qa_dataset[0]['question']
ground_truth_id = qa_dataset[0]['relevant_passage_id']

print(f"\nQuestion: {test_question}")
print(f"We are hoping to retrieve Document ID: {ground_truth_id}\n")

# 5. Search!
retrieved_chunks = retriever.retrieve(test_question)

for i, chunk in enumerate(retrieved_chunks):
    # Remember we saved the "id" in the metadata earlier!
    retrieved_id = chunk.metadata.get('id', 'Unknown')
    print(f"Rank {i+1} - Retrieved ID: {retrieved_id} (Score: {chunk.score:.3f})")

Setting up embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3285.14it/s]


LLM is explicitly disabled. Using MockLLM.
Building the Vector Index (This will take a minute or two)...
Index built successfully!

Question: Bob and Ted are close friends. Ted is on trial for drug offences, and Bob has been selected as a juror in Ted’s case. Is the judge required to excuse Bob from serving on the jury?
We are hoping to retrieve Document ID: 1.2-c2-s2

Rank 1 - Retrieved ID: 1.2-c2-s2 (Score: 0.716)
Rank 2 - Retrieved ID: 1.2.1-c3-s1 (Score: 0.714)
Rank 3 - Retrieved ID: 1.2-c2-s3 (Score: 0.708)


In [9]:
import qdrant_client

QDRANT_URL = "https://qdrant.mittelstand-zukunftslab.de"
QDRANT_KEY = "zeuhzyfpmfyp7i71opfbglk9cwyg7uhd"

print("Pinging Qdrant Database...")
try:
    # We added timeout=60 (seconds) so it waits longer
    client = qdrant_client.QdrantClient(url=QDRANT_URL, api_key=QDRANT_KEY, timeout=60)

    # Just ask Qdrant what collections (folders) it currently has
    collections = client.get_collections()
    print("SUCCESS! Connected to Qdrant.")
    print("Current collections in database:", collections)
except Exception as e:
    print(f"FAILED to connect to Qdrant. Error: {e}")

Pinging Qdrant Database...
FAILED to connect to Qdrant. Error: [WinError 10061] No connection could be made because the target machine actively refused it


In [10]:
import qdrant_client

QDRANT_URL = "https://qdrant.mittelstand-zukunftslab.de"
QDRANT_KEY = "zeuhzyfpmfyp7i71opfbglk9cwyg7uhd"

print("Pinging Qdrant Database (Attempt 2)...")
try:
    # We explicitly add port=443 to force it through the standard HTTPS door
    client = qdrant_client.QdrantClient(
        url=QDRANT_URL,
        port=443,
        api_key=QDRANT_KEY,
        timeout=60
    )

    collections = client.get_collections()
    print("SUCCESS! Connected to Qdrant.")
    print("Current collections in database:", collections)
except Exception as e:
    print(f"FAILED to connect to Qdrant. Error: {e}")

Pinging Qdrant Database (Attempt 2)...
SUCCESS! Connected to Qdrant.
Current collections in database: collections=[]


In [16]:
import requests

LITELLM_URL = "https://litellm.mittelstand-zukunftslab.de/"
LITELLM_KEY = "sk-pfrgLhvL2Idn3HCWC0yPAQ"

url = LITELLM_URL.rstrip('/') + "/v1/models"
headers = {"Authorization": f"Bearer {LITELLM_KEY}"}

print("Searching Fadel's server for the new Qwen model...")
response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    print("Found these Qwen / Embedding models:\n")
    for model in data.get('data', []):
        model_id = model['id']
        # Look for anything with 'qwen' or 'embed'
        if 'embed' in model_id.lower() or 'qwen' in model_id.lower():
            print(f" -> {model_id}")
else:
    print(f"FAILED! Error {response.status_code}: {response.text}")

Searching Fadel's server for the new Qwen model...
Found these Qwen / Embedding models:

 -> qwen/qwen3-embedding-8b
 -> qwen/qwen3-embedding-4b
 -> openrouter/qwen/qwen-vl-plus
 -> openrouter/qwen/qwen3.5-plus-02-15
 -> openrouter/qwen/qwen-2.5-coder-32b-instruct
 -> openrouter/qwen/qwen3-235b-a22b-2507
 -> openrouter/qwen/qwen3-235b-a22b-thinking-2507
 -> openrouter/qwen/qwen3.5-27b
 -> openrouter/qwen/qwen3.5-122b-a10b
 -> openrouter/qwen/qwen3.5-flash-02-23
 -> openrouter/qwen/qwen3.5-397b-a17b
 -> openrouter/qwen/qwen3.6-plus
 -> openrouter/qwen/qwen3-coder-plus
 -> openrouter/qwen/qwen3-coder
 -> openrouter/qwen/qwen3.5-35b-a3b


In [25]:
import requests

LITELLM_URL = "https://litellm.mittelstand-zukunftslab.de/"
LITELLM_KEY = "sk-pfrgLhvL2Idn3HCWC0yPAQ"

print("Asking Fadel's server directly for the vector size...")

# We hit the standard OpenAI-compatible embeddings endpoint
url = LITELLM_URL.rstrip('/') + "/v1/embeddings"
headers = {
    "Authorization": f"Bearer {LITELLM_KEY}",
    "Content-Type": "application/json"
}

# The raw payload exactly as Fadel's server expects it
payload = {
    "model": "qwen/qwen3-embedding-8b",
    "input": "test"
}

response = requests.post(url, headers=headers, json=payload)

if response.status_code == 200:
    data = response.json()
    # Extract the math array from the JSON response
    vector = data['data'][0]['embedding']
    print(f"\nSUCCESS! The Vector Size (Dimensions) for Qwen3-8B is: {len(vector)}")
else:
    print(f"FAILED! Error {response.status_code}: {response.text}")

Asking Fadel's server directly for the vector size...

SUCCESS! The Vector Size (Dimensions) for Qwen3-8B is: 4096


In [32]:
import os
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.litellm import LiteLLMEmbedding

LITELLM_URL = "https://litellm.mittelstand-zukunftslab.de/"
LITELLM_KEY = "sk-pfrgLhvL2Idn3HCWC0yPAQ"

print("1. Setting up Cloud Embedding Model (Qwen3-8B) - NOW FIXED BY FADEL!")
Settings.embed_model = LiteLLMEmbedding(
    model_name="openai/qwen/qwen3-embedding-8b",
    api_key=LITELLM_KEY,
    api_base=LITELLM_URL,
    embed_batch_size=10 # Safe mode batching
)
Settings.llm = None

print("2. Building the Vector Database locally in RAM...")
# This will now send the text to Qwen over the internet to get the math vectors back!
index = VectorStoreIndex.from_documents(
    llama_documents,
    show_progress=True,
    insert_batch_size=10
)

print("\n3. Saving the database to your hard drive...")
save_path = "../data/processed/agrim_legal_bench_index"
os.makedirs(save_path, exist_ok=True)
index.storage_context.persist(persist_dir=save_path)

print(f"SUCCESS! Notebook complete. The RAG ingestion pipeline is saved locally to: {save_path}")

1. Setting up Cloud Embedding Model (Qwen3-8B) - NOW FIXED BY FADEL!
LLM is explicitly disabled. Using MockLLM.
2. Building the Vector Database locally in RAM...






Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]



Applying transformations: 100%|██████████| 1/1 [00:00<00:00,  1.12it/s]




Generating embeddings:   0%|          | 0/10 [00:00<?, ?it/s]



Generating embeddings: 100%|██████████| 10/10 [00:02<00:00,  4.81it/s]




Generating embeddings:   0%|          | 0/10 [00:00<?, ?it/s]



Generating embeddings: 100%|██████████| 10/10 [00:01<00:00,  5.22it/s]




Generating embeddings:   0%|          | 0/10 [00:00<?, ?it/s]



Generating embeddings: 100%|██████████| 10/10 [00:01<00:00,  5.79it/s]




Generating embeddings:   0%|          | 0/10 [00:00<?, ?it/s]



Generating embeddings: 100%|██████████| 10/10 [00:01<00:00,  6.39it/s]




Generating embeddings:   0%|          | 0/10 [00:00<?, ?it/s]



Generating embeddings: 100%|██████████| 10/10 [00:11<00:00,  1.12s/it]




Generating embeddings:   0%|          | 0/10 [00:00<?, ?it/s]



Generating embeddings: 100%|██████████| 10/10 [00:11<00:00,  1.18s/it]




Generating em


3. Saving the database to your hard drive...
SUCCESS! Notebook complete. The RAG ingestion pipeline is saved locally to: ../data/processed/agrim_legal_bench_index


In [36]:
import time
import httpx
from llama_index.core import StorageContext, load_index_from_storage, Settings
from llama_index.embeddings.litellm import LiteLLMEmbedding

# --- CONFIG ---
LITELLM_URL = "https://litellm.mittelstand-zukunftslab.de/"
LITELLM_KEY = "sk-pfrgLhvL2Idn3HCWC0yPAQ"

QDRANT_URL = "https://qdrant.mittelstand-zukunftslab.de"
QDRANT_KEY = "zeuhzyfpmfyp7i71opfbglk9cwyg7uhd"

COLLECTION_NAME = "agrim_legal_bench_v1"
SAVE_PATH = "../data/processed/agrim_legal_bench_index"
BATCH_SIZE = 50
MAX_RETRIES = 8
RETRY_DELAY = 5

Settings.embed_model = LiteLLMEmbedding(
    model_name="openai/qwen/qwen3-embedding-8b",
    api_key=LITELLM_KEY,
    api_base=LITELLM_URL
)
Settings.llm = None

print("1. Loading locally saved Qwen embeddings from your hard drive...")
local_storage_context = StorageContext.from_defaults(persist_dir=SAVE_PATH)
local_index = load_index_from_storage(local_storage_context)
nodes = list(local_index.docstore.docs.values())
print(f"Loaded {len(nodes)} pre-computed chunks!")

vector_store = local_storage_context.vector_store
embedding_dict = None
for attr_path in ["data.embedding_dict", "_data.embedding_dict"]:
    obj = vector_store
    ok = True
    for part in attr_path.split("."):
        if hasattr(obj, part):
            obj = getattr(obj, part)
        else:
            ok = False
            break
    if ok:
        embedding_dict = obj
        break

if embedding_dict is None:
    raise RuntimeError(
        "Could not locate embedding_dict on vector_store. "
        f"Available attrs: {dir(vector_store)}"
    )

print(f"Found {len(embedding_dict)} embeddings in vector store.")

session = httpx.Client(
    base_url=QDRANT_URL,
    headers={"api-key": QDRANT_KEY, "Content-Type": "application/json"},
    timeout=60
)

print("\n2. Checking Qdrant collection...")
r = session.get(f"/collections/{COLLECTION_NAME}")
print(f"   Status: {r.status_code}")
if r.status_code != 200:
    raise RuntimeError(f"Collection check failed: {r.text}")
print("   Collection found!")

print(f"\n3. Uploading {len(nodes)} points in batches of {BATCH_SIZE}...")

point_id_counter = 0
success_count = 0

for i in range(0, len(nodes), BATCH_SIZE):
    batch = nodes[i: i + BATCH_SIZE]
    points = []
    for node in batch:
        node_id = node.node_id
        embedding = embedding_dict.get(node_id)
        if embedding is None:
            print(f"    Skipping node {node_id} -- no embedding found.")
            continue
        payload = {
            "text": node.get_content(),
            **(node.metadata or {})
        }
        points.append({
            "id": point_id_counter,
            "vector": embedding,
            "payload": payload
        })
        point_id_counter += 1

    if not points:
        continue

    label = f"{i}-{i + len(batch)}"
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.put(
                f"/collections/{COLLECTION_NAME}/points",
                json={"points": points},
                params={"wait": "true"}
            )
            if resp.status_code in (200, 202):
                print(f" -> Batch {label}: SUCCESS ({len(points)} points)")
                success_count += len(points)
                break
            else:
                print(f" -> Batch {label} attempt {attempt} failed: {resp.status_code} {resp.text[:200]}")
                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_DELAY)
        except Exception as e:
            print(f" -> Batch {label} attempt {attempt} exception: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
    time.sleep(0.3)

print(f"\nDONE! Uploaded {success_count} / {len(nodes)} points to Qdrant.")

LLM is explicitly disabled. Using MockLLM.
1. Loading locally saved Qwen embeddings from your hard drive...
Loaded 4876 pre-computed chunks!
Found 4876 embeddings in vector store.

2. Checking Qdrant collection...
   Status: 200
   Collection found!

3. Uploading 4876 points in batches of 50...
 -> Batch 0-50: SUCCESS (50 points)
 -> Batch 50-100: SUCCESS (50 points)
 -> Batch 100-150: SUCCESS (50 points)
 -> Batch 150-200: SUCCESS (50 points)
 -> Batch 200-250: SUCCESS (50 points)
 -> Batch 250-300: SUCCESS (50 points)
 -> Batch 300-350: SUCCESS (50 points)
 -> Batch 350-400: SUCCESS (50 points)
 -> Batch 400-450: SUCCESS (50 points)
 -> Batch 450-500: SUCCESS (50 points)
 -> Batch 500-550: SUCCESS (50 points)
 -> Batch 550-600: SUCCESS (50 points)
 -> Batch 600-650: SUCCESS (50 points)
 -> Batch 650-700: SUCCESS (50 points)
 -> Batch 700-750: SUCCESS (50 points)
 -> Batch 750-800: SUCCESS (50 points)
 -> Batch 800-850: SUCCESS (50 points)
 -> Batch 850-900: SUCCESS (50 points)
 -> Ba

In [44]:
import httpx
import litellm

litellm.drop_params = True

# --- CONFIG ---
LITELLM_URL = "https://litellm.mittelstand-zukunftslab.de"
LITELLM_KEY = "sk-pfrgLhvL2Idn3HCWC0yPAQ"
QDRANT_URL = "https://qdrant.mittelstand-zukunftslab.de"
QDRANT_KEY = "zeuhzyfpmfyp7i71opfbglk9cwyg7uhd"
COLLECTION_NAME = "agrim_legal_bench_v1"
TOP_K = 3

qdrant_session = httpx.Client(
    base_url=QDRANT_URL,
    headers={"api-key": QDRANT_KEY, "Content-Type": "application/json"},
    timeout=30
)

test_question = (
    "Bob and Ted are close friends. Ted is on trial for drug offences, and Bob "
    "has been selected as a juror in Ted's case. Is the judge required to "
    "excuse Bob from serving on the jury?"
)
print(f"--- USER QUESTION ---\n{test_question}")

# --- 1. Embed the question via LiteLLM (raw, no llama-index wrapper) ---
print("\n1. Embedding the question...")
embed_response = litellm.embedding(
    model="openai/qwen/qwen3-embedding-8b",
    input=[test_question],
    api_key=LITELLM_KEY,
    api_base=LITELLM_URL,
    timeout=30
)
query_vector = embed_response.data[0]["embedding"]
print(f"   Got embedding vector of length {len(query_vector)}.")

# --- 2. Search Qdrant directly via REST (bypassing qdrant-client SDK) ---
print(f"\n2. Searching Qdrant for top {TOP_K} matches...")
search_resp = qdrant_session.post(
    f"/collections/{COLLECTION_NAME}/points/search",
    json={
        "vector": query_vector,
        "limit": TOP_K,
        "with_payload": True
    }
)
search_resp.raise_for_status()
results = search_resp.json()["result"]
print(f"   Retrieved {len(results)} documents.")

context_chunks = []
for i, r in enumerate(results):
    text = r["payload"].get("text", "")
    score = r["score"]
    print(f"   [{i+1}] score={score:.4f} -- {text[:80]}...")
    context_chunks.append(text)

context_str = "\n\n---\n\n".join(context_chunks)

# --- 3. Generate an answer via LiteLLM chat completion ---
print("\n3. Generating answer with Qwen 3.6 Plus...")
prompt = f"""Answer the question using ONLY the context below. If the context doesn't contain the answer, say so.

Context:
{context_str}

Question: {test_question}

Answer:"""

chat_response = litellm.completion(
    model="openai/openrouter/qwen/qwen3.6-plus",  # <- fixed
    messages=[{"role": "user", "content": prompt}],
    api_key=LITELLM_KEY,
    api_base=LITELLM_URL,
    timeout=60
)

answer = chat_response.choices[0].message.content
print(f"\n--- AI ANSWER ---\n{answer}")


--- USER QUESTION ---
Bob and Ted are close friends. Ted is on trial for drug offences, and Bob has been selected as a juror in Ted's case. Is the judge required to excuse Bob from serving on the jury?

1. Embedding the question...
   Got embedding vector of length 4096.

2. Searching Qdrant for top 3 matches...
   Retrieved 3 documents.
   [1] score=0.6284 -- ## Excusing Jurors...
   [2] score=0.6220 -- Last updated: 11 July 2018...
   [3] score=0.6190 -- # 1.2.1 Charge: Jury Empanelment

Members of the jury panel, in a moment I am go...

3. Generating answer with Qwen 3.6 Plus...

--- AI ANSWER ---
Based on the provided context, a juror who knows someone involved in the trial may not be able to remain unbiased, and retaining them could create an appearance of bias that would "undermine the interests of justice." The context instructs such jurors to "apply to be excused." However, the context does not explicitly state that the judge is strictly *required* to excuse the juror; it only 